# Kathakaar — Cultural Narration Style model (QLoRA)

Fine-tune a small open instruct LLM (**Qwen2.5-3B-Instruct**) with **QLoRA** so it retells
grounded facts in different **storytelling traditions** (oral, griot, ballad, koan, mythic…).
Runs on a **free T4** (Colab/Kaggle). The model adds the *cadence and form* of a tradition —
never new facts — so it stays compatible with Kathakaar's provenance rule.

### The real work is the dataset (read this)
- A model learns a tradition only from authentic examples of it. The ~10 seed pairs here are a
  **starter/format demo** — you'll get real results by growing each tradition to 30–100 pairs
  of `(facts → styled retelling)` from openly-licensed / public-domain sources.
- **These traditions belong to living communities.** A griot is a hereditary West African role;
  dastangoi, kamishibai and others are real lineages. Source respectfully, attribute, label
  outputs as *interpretations in the style of*, and credit practitioners where you can. That
  care is the integrity of the project — the same provenance ethic, applied to form.

> Colab: **Runtime → Change runtime type → T4 GPU**, then run top to bottom.

## 1. GPU + install

In [ ]:
!nvidia-smi
!pip -q install "transformers>=4.44" "trl>=0.9" "peft>=0.12" "datasets>=2.20" \
    "accelerate>=0.33" bitsandbytes
print('\ninstalled')

## 2. Load the dataset
Upload `narration_style_dataset.jsonl` (the seed file) into this notebook's working dir, or
point to your own larger file. Each line: `{style, style_label, facts, narration}`.

In [ ]:
from datasets import load_dataset
ds = load_dataset('json', data_files='narration_style_dataset.jsonl', split='train')
print(ds)
print('\nexample:\n', ds[0])

## 3. Load the base model in 4-bit + LoRA config

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig

MODEL = 'Qwen/Qwen2.5-3B-Instruct'  # open, ungated; or a newer small instruct model

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map='auto')
model.config.use_cache = False

peft_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                      task_type='CAUSAL_LM',
                      target_modules=['q_proj','k_proj','v_proj','o_proj',
                                      'gate_proj','up_proj','down_proj'])

## 4. Format examples with the chat template

In [ ]:
SYSTEM = ('You are Kathakaar, a cultural storyteller. Retell the given grounded facts in the '
          'requested storytelling tradition. Stay faithful to the facts; add only the cadence, '
          'motifs, and framing of the tradition — never invent new facts.')

def to_text(ex):
    msgs = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': f"Tradition: {ex['style_label']}\nFacts: {ex['facts']}\n\nTell it in this tradition."},
        {'role': 'assistant', 'content': ex['narration']},
    ]
    return {'text': tok.apply_chat_template(msgs, tokenize=False)}

train_ds = ds.map(to_text, remove_columns=ds.column_names)
print(train_ds[0]['text'][:500])

## 5. Train (QLoRA)
With a tiny seed set we use several epochs so the style is learned; **watch for overfitting**
and grow the dataset for real quality. On a T4 this is minutes.

In [ ]:
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir='/content/kathakaar-narration-lora',
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    num_train_epochs=8, learning_rate=2e-4, lr_scheduler_type='cosine',
    warmup_ratio=0.05, logging_steps=5, save_steps=50,
    fp16=True, max_seq_length=1024, packing=False, report_to='none')

trainer = SFTTrainer(model=model, train_dataset=train_ds, peft_config=peft_cfg,
                     args=cfg, dataset_text_field='text', tokenizer=tok)
trainer.train()
trainer.save_model('/content/kathakaar-narration-lora')
print('\nadapter saved.')

## 6. Try it — same facts, different traditions
This is the test that matters: hold the facts constant, change only the tradition.

In [ ]:
def narrate(facts, style_label, max_new_tokens=200):
    msgs = [{'role': 'system', 'content': SYSTEM},
            {'role': 'user', 'content': f'Tradition: {style_label}\nFacts: {facts}\n\nTell it in this tradition.'}]
    inp = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(model.device)
    out = model.generate(inp, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.85, top_p=0.9)
    return tok.decode(out[0][inp.shape[1]:], skip_special_tokens=True).strip()

facts = ('The Great Wall of China was built across northern China over many centuries; sections '
         'near Beijing were rebuilt during the Ming dynasty; it served for border defence and '
         'control of trade routes.')

for style in ['Griot (West Africa)', 'Koan / Parable (East Asia)', 'Mythic Cycle (Norse / Hellenic)']:
    print('=' * 8, style, '=' * 8)
    print(narrate(facts, style), '\n')

## 7. Plug it back into Kathakaar
Replace (or augment) the hand-written templates in `studio/app/cinematic.py` with this model:

1. Serve the adapter on a GPU host (e.g. **vLLM**, HF Inference Endpoint, or `text-generation`).
2. In `cinematic.py`, for each scene call the model with the **grounded `caption`** (the cited
   fact) + the chosen tradition → use the returned text as the scene `narration`.
3. Keep the refusal gate and citations exactly as they are — you only feed it facts that the
   sources prove, so the styling never introduces ungrounded claims. Gate it behind a
   `NARRATION_MODEL_URL` env var with the current template version as the free fallback.

That upgrade turns your five fixed forms into an open-ended, learnable set of traditions —
while preserving the 0.00 false-accept guarantee.

## 8. Grow the dataset (the actual project)
- Aim for **30–100 pairs per tradition**, balanced across traditions and places.
- Hold out ~10% as a **test set**; evaluate by (a) human rating of authenticity and (b) a
  style classifier that checks the output matches the requested tradition.
- Keep a `source` + `license` field per example; prefer public-domain anthologies and openly
  licensed texts; record attribution.
- Add a **faithfulness check**: verify the styled output introduces no facts absent from the
  input (your grounding ethic, now measured for narration too — a great evaluation to publish).